# AlphaZero для Тоғыз Құмалақ 🎮

Обучение AlphaZero играть в казахскую настольную игру Тоғыз Құмалақ.

**Требования:** Google Colab с GPU (A100 рекомендуется)

## Содержание
1. Установка и настройка
2. Обучение модели
3. Мониторинг метрик обучения
4. Игра против обученной модели

## 1. Установка и настройка

In [ ]:
# Проверяем GPU
!nvidia-smi
import torch
print(f'\nPyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Клонируем репозиторий
!git clone https://github.com/Eraly-ml/alpha_zero_toguz_Qumalaq.git
%cd alpha_zero_toguz_Qumalaq

In [ ]:
# Устанавливаем зависимости
!pip install -r requirements.txt

In [ ]:
# Проверяем что env работает
import sys
sys.path.insert(0, '.')
from alpha_zero.envs.toguz import ToguzKumalakEnv

env = ToguzKumalakEnv(num_stack=4)
obs = env.reset()
print(f'Observation shape: {obs.shape}')
print(f'Action space: {env.action_space}')
print(f'Legal actions: {env.legal_actions}')
env.render()
print('\nEnvironment OK!')

In [ ]:
# Проверяем что сеть работает с нашим env
from alpha_zero.core.network import AlphaZeroNet
import numpy as np

input_shape = env.observation_space.shape
num_actions = env.action_space.n
print(f'Input shape: {input_shape}, Num actions: {num_actions}')

net = AlphaZeroNet(input_shape, num_actions, num_res_block=6, num_filters=64, num_fc_units=128)
x = torch.from_numpy(obs[None, ...]).float()
pi_logits, value = net(x)
print(f'Policy logits shape: {pi_logits.shape}')
print(f'Value shape: {value.shape}')
print(f'Network parameters: {sum(p.numel() for p in net.parameters()):,}')
print('\nNetwork OK!')

## 2. Обучение модели

### Параметры обучения
| Параметр | Значение | Описание |
|----------|----------|----------|
| num_actors | 8 | Количество процессов self-play |
| num_simulations | 200 | MCTS симуляций на ход |
| num_res_blocks | 6 | Residual блоков в сети |
| num_filters | 64 | Фильтров в conv слоях |
| max_training_steps | 100,000 | Шагов обучения |
| batch_size | 256 | Размер батча |

In [ ]:
# Запускаем обучение
# Для A100 Colab эти параметры оптимальны
# Уменьшите num_actors если мало CPU ядер

!python -m alpha_zero.training_toguz \
    --num_stack=4 \
    --num_res_blocks=6 \
    --num_filters=64 \
    --num_fc_units=128 \
    --num_actors=8 \
    --num_simulations=200 \
    --num_parallel=4 \
    --batch_size=256 \
    --replay_capacity=500000 \
    --min_games=1000 \
    --games_per_ckpt=1000 \
    --max_training_steps=100000 \
    --init_lr=0.01 \
    --lr_milestones=50000 \
    --lr_milestones=80000 \
    --ckpt_interval=500 \
    --log_interval=100 \
    --ckpt_dir=./checkpoints/toguz \
    --logs_dir=./logs/toguz \
    --save_sgf_dir=./games/selfplay_games/toguz \
    --seed=1

## 3. Мониторинг метрик обучения

Визуализация статистик обучения: потери (policy loss, value loss), длина игр, ELO рейтинг, win rate.

In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import math
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

LOGS_DIR = './logs/toguz'

def shorten(value, tick_number=None):
    num_thousands = 0 if abs(value) < 1000 else math.floor(math.log10(abs(value)) / 3)
    value = round(value / 1000**num_thousands, 2)
    return f'{value:g}' + ' KMGTPEZY'[num_thousands]


def load_learner_csv(logs_dir):
    """Load learner training statistics."""
    csv_file = os.path.join(logs_dir, 'learner.csv')
    if os.path.exists(csv_file):
        return pd.read_csv(csv_file)
    return None


def load_actor_csvs(logs_dir):
    """Load all actor self-play statistics."""
    actor_files = glob.glob(os.path.join(logs_dir, 'actor*.csv'))
    if not actor_files:
        return None
    return pd.concat([pd.read_csv(f) for f in actor_files], sort=True)


def load_eval_csv(logs_dir):
    """Load evaluation statistics."""
    csv_file = os.path.join(logs_dir, 'evaluator.csv')
    if os.path.exists(csv_file):
        return pd.read_csv(csv_file)
    return None


print(f'Looking for logs in: {LOGS_DIR}')
print(f'Files found: {os.listdir(LOGS_DIR) if os.path.exists(LOGS_DIR) else "directory not found"}')

In [ ]:
# === TRAINING LOSSES ===
learner_df = load_learner_csv(LOGS_DIR)

if learner_df is not None and len(learner_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Policy loss
    ax = axes[0]
    ax.plot(learner_df['training_steps'], learner_df['policy_loss'], linewidth=0.8, color='#2196F3')
    ax.set_title('Policy Loss')
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Cross-Entropy Loss')
    ax.xaxis.set_major_formatter(FuncFormatter(shorten))

    # Value loss
    ax = axes[1]
    ax.plot(learner_df['training_steps'], learner_df['value_loss'], linewidth=0.8, color='#FF5722')
    ax.set_title('Value Loss')
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('MSE Loss')
    ax.xaxis.set_major_formatter(FuncFormatter(shorten))

    # Learning rate
    ax = axes[2]
    if 'learning_rate' in learner_df.columns:
        ax.plot(learner_df['training_steps'], learner_df['learning_rate'], linewidth=0.8, color='#4CAF50')
    ax.set_title('Learning Rate')
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('LR')
    ax.xaxis.set_major_formatter(FuncFormatter(shorten))

    plt.tight_layout()
    plt.savefig('./logs/toguz/training_losses.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Latest training step: {learner_df["training_steps"].max()}')
    print(f'Latest policy loss: {learner_df["policy_loss"].iloc[-1]:.4f}')
    print(f'Latest value loss: {learner_df["value_loss"].iloc[-1]:.4f}')
else:
    print('No learner data found yet. Start training first.')

In [ ]:
# === SELF-PLAY STATISTICS ===
actor_df = load_actor_csvs(LOGS_DIR)

if actor_df is not None and len(actor_df) > 0:
    # Derive columns
    actor_df['game_count'] = 1
    actor_df['black_won'] = actor_df['game_result'].apply(
        lambda x: 1 if isinstance(x, str) and x.startswith('B') else 0
    )
    actor_df['white_won'] = actor_df['game_result'].apply(
        lambda x: 1 if isinstance(x, str) and x.startswith('W') else 0
    )
    actor_df['draw'] = actor_df['game_result'].apply(
        lambda x: 1 if isinstance(x, str) and x == 'DRAW' else 0
    )

    # Group by training step for smoother plots
    grouped = actor_df.groupby('training_step').agg({
        'game_count': 'sum',
        'game_length': 'mean',
        'black_won': 'sum',
        'white_won': 'sum',
        'draw': 'sum',
    }).reset_index()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Game length distribution
    ax = axes[0]
    ax.plot(grouped['training_step'], grouped['game_length'], linewidth=0.8, color='#9C27B0')
    ax.set_title('Average Game Length')
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Moves per Game')
    ax.xaxis.set_major_formatter(FuncFormatter(shorten))

    # Cumulative games
    ax = axes[1]
    ax.plot(grouped['training_step'], grouped['game_count'].cumsum(), linewidth=0.8, color='#FF9800')
    ax.set_title('Total Self-Play Games')
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Cumulative Games')
    ax.xaxis.set_major_formatter(FuncFormatter(shorten))

    # Win rates
    ax = axes[2]
    total_games = grouped['black_won'] + grouped['white_won'] + grouped['draw']
    total_games = total_games.replace(0, 1)  # avoid division by zero
    ax.plot(grouped['training_step'], grouped['black_won'] / total_games * 100,
            linewidth=0.8, label='P1 (Black) Win %', color='#333333')
    ax.plot(grouped['training_step'], grouped['white_won'] / total_games * 100,
            linewidth=0.8, label='P2 (White) Win %', color='#BDBDBD')
    ax.plot(grouped['training_step'], grouped['draw'] / total_games * 100,
            linewidth=0.8, label='Draw %', color='#03A9F4', linestyle='--')
    ax.set_title('Win Rate by Player')
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Win Rate (%)')
    ax.legend()
    ax.xaxis.set_major_formatter(FuncFormatter(shorten))

    plt.tight_layout()
    plt.savefig('./logs/toguz/selfplay_stats.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Total self-play games: {actor_df["game_count"].sum()}')
    print(f'Average game length: {actor_df["game_length"].mean():.1f} moves')
    print(f'P1 wins: {actor_df["black_won"].sum()}, P2 wins: {actor_df["white_won"].sum()}, Draws: {actor_df["draw"].sum()}')
else:
    print('No self-play data found yet. Start training first.')

In [ ]:
# === ELO RATING PROGRESS ===
eval_df = load_eval_csv(LOGS_DIR)

if eval_df is not None and len(eval_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ELO rating over time
    ax = axes[0]
    if 'black_elo' in eval_df.columns:
        ax.plot(eval_df.index, eval_df['black_elo'], linewidth=1.2, color='#E91E63', marker='o', markersize=3)
    ax.set_title('ELO Rating Progress')
    ax.set_xlabel('Evaluation Round')
    ax.set_ylabel('ELO Rating')
    ax.grid(True, alpha=0.3)

    # New model win rate vs previous
    ax = axes[1]
    if 'new_model_wins' in eval_df.columns and 'games_played' in eval_df.columns:
        win_rate = eval_df['new_model_wins'] / eval_df['games_played'].replace(0, 1) * 100
        ax.plot(eval_df.index, win_rate, linewidth=1.2, color='#4CAF50', marker='o', markersize=3)
        ax.axhline(y=55, color='red', linestyle='--', alpha=0.5, label='55% threshold')
        ax.legend()
    ax.set_title('New Model Win Rate vs Previous')
    ax.set_xlabel('Evaluation Round')
    ax.set_ylabel('Win Rate (%)')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('./logs/toguz/elo_progress.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No evaluation data found yet. Training needs to progress further.')

In [ ]:
# === TRAINING SUMMARY ===
import os

ckpt_dir = './checkpoints/toguz'
if os.path.exists(ckpt_dir):
    ckpts = sorted([f for f in os.listdir(ckpt_dir) if f.endswith('.ckpt')])
    print(f'Checkpoints saved: {len(ckpts)}')
    if ckpts:
        print(f'Latest: {ckpts[-1]}')
        print(f'All checkpoints:')
        for c in ckpts:
            size_mb = os.path.getsize(os.path.join(ckpt_dir, c)) / 1e6
            print(f'  {c} ({size_mb:.1f} MB)')
else:
    print('No checkpoints found yet.')

## 4. Игра против обученной модели

Выберите чекпоинт и сыграйте против AI в терминале.

In [ ]:
# Запустить игру против AI
# Измените путь к чекпоинту на нужный

import glob
ckpts = sorted(glob.glob('./checkpoints/toguz/*.ckpt'))
if ckpts:
    best_ckpt = ckpts[-1]
    print(f'Using checkpoint: {best_ckpt}')
    print('Starting game...')
    print('Enter pit number 1-9 to make a move, q to quit')
    !python eval_play/eval_agent_toguz_cmd.py \
        --human_vs_ai=True \
        --human_color=black \
        --white_ckpt={best_ckpt} \
        --num_simulations=200
else:
    print('No checkpoints found. Train the model first.')
    print('You can still play against a random model:')
    !python eval_play/eval_agent_toguz_cmd.py \
        --human_vs_ai=True \
        --human_color=black \
        --white_ckpt='' \
        --num_simulations=50

In [ ]:
# AI vs AI - смотреть как модели играют друг с другом
import torch
import sys
sys.path.insert(0, '.')

from alpha_zero.envs.toguz import ToguzKumalakEnv
from alpha_zero.core.network import AlphaZeroNet
from alpha_zero.core.pipeline import create_mcts_player, set_seed, disable_auto_grad

set_seed(42)
env = ToguzKumalakEnv(num_stack=4)

input_shape = env.observation_space.shape
num_actions = env.action_space.n

# Load network (random weights if no checkpoint)
import glob, os
ckpts = sorted(glob.glob('./checkpoints/toguz/*.ckpt'))

net = AlphaZeroNet(input_shape, num_actions, 6, 64, 128)
if ckpts:
    state = torch.load(ckpts[-1], map_location='cpu')
    net.load_state_dict(state['network'])
    print(f'Loaded: {ckpts[-1]}')
else:
    print('No checkpoint found, using random weights')

disable_auto_grad(net)
net.eval()

player = create_mcts_player(net, 'cpu', num_simulations=100, num_parallel=4)

# Play a demo game
env.reset()
env.render()

while not env.is_game_over():
    move, *_ = player(env, None, 19652, 1.25)
    who = 'P1' if env.to_play == env.black_player else 'P2'
    env.step(move)
    print(f'\nStep {env.steps}: {who} plays pit {move + 1}')
    env.render()

print(f'\nGame over! Result: {env.get_result_string()}')
print(f'Kazans: P1={env.kazans[0]}, P2={env.kazans[1]}')
print(f'Total moves: {env.steps}')

## 5. Сохранение модели в Google Drive (опционально)

In [ ]:
# Монтируем Google Drive для сохранения чекпоинтов
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/toguz_kumalak_checkpoints
!cp -r ./checkpoints/toguz/* /content/drive/MyDrive/toguz_kumalak_checkpoints/
!cp -r ./logs/toguz/* /content/drive/MyDrive/toguz_kumalak_checkpoints/
print('Checkpoints and logs saved to Google Drive!')